# Sync Location Curations

Syncs location curations of open-access fields (`is_oa`, `pdf_url`, `license`; oxjob #1170) from the users Heroku Postgres database to a local Databricks Delta table. Runs in `walden_end2end.yaml` before `Apply_Location_Curations`, which lays them over `locations_mapped` every night (last one wins per location and field, together with the legacy `openalex.curations.approved_curations`).

**Source**: `openalex_users.public.location_curations`, a typed view over `curations` (users-api migration 115): one row per curation, `location_id` = the API's location id (`<native_id_namespace>:<native_id>`).

**Target**: `openalex.works.location_curations`

A twin curation (the licence <-> `is_oa` rule, minted by users-api in the same request) has `user_id` NULL and `caused_by` = the curation that caused it. Location curations are never deleted through the API (`DELETE /curations/{id}` refuses them); rows leave only when users-api deletes an account (oxjob #1436), through `NOT MATCHED BY SOURCE`, and the guardrail refuses that clause on a suspicious empty or short source.

## Create target table (idempotent)

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS openalex.works.location_curations (
    curation_id        STRING NOT NULL,
    location_id        STRING NOT NULL,
    property           STRING NOT NULL,
    action             STRING NOT NULL,
    value              STRING,
    user_id            STRING,  -- NULL on twin curations
    caused_by          STRING,  -- twin curations: the curation that caused this one
    created            TIMESTAMP,
    updated_datetime   TIMESTAMP
)
USING DELTA
CLUSTER BY (location_id)

## Sync location curations

In [ ]:
%sql
-- Empty-when-target-nonempty fails unconditionally; decline check is overridable.
-- (Both 0 on startup is legitimate and doesn't fail.)
DECLARE OR REPLACE VARIABLE new_count BIGINT;
DECLARE OR REPLACE VARIABLE current_count BIGINT;
DECLARE OR REPLACE VARIABLE allowed_decline BIGINT DEFAULT 10;

SET VARIABLE new_count = (SELECT COUNT(*) FROM openalex_users.public.location_curations);
SET VARIABLE current_count = (SELECT COUNT(*) FROM openalex.works.location_curations);

SELECT
  new_count AS new_curations,
  current_count AS current_curations,
  openalex.works.e2e_override_active('guardrails_override', :guardrails_override) AS guardrails_override;  -- job parameter OR a pre-cleared row in openalex.works.e2e_overrides

SELECT CASE
  WHEN current_count > 0 AND new_count = 0
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source has 0 location curations but the target holds ',
    CAST(current_count AS STRING),
    ' rows. Aborting to prevent mass delete. Investigate the source view before re-running.'))
END;

SELECT CASE
  WHEN current_count > 0
   AND new_count < current_count - allowed_decline
   AND NOT openalex.works.e2e_override_active('guardrails_override', :guardrails_override)
  THEN RAISE_ERROR(CONCAT(
    'GUARDRAIL FAILED: source location curation count declined by ',
    CAST(current_count - new_count AS STRING),
    ' (', CAST(new_count AS STRING), ' vs ', CAST(current_count AS STRING),
    '), exceeding allowed decline of ', CAST(allowed_decline AS STRING),
    '. Set guardrails_override=true on the job, or pre-clear the run in openalex.works.e2e_overrides, to bypass.'))
END;

In [ ]:
%sql
MERGE INTO openalex.works.location_curations AS target
USING (
    SELECT
        curation_id, location_id, property, action, value, user_id, caused_by, created,
        CURRENT_TIMESTAMP() AS updated_datetime
    FROM openalex_users.public.location_curations
) AS source
ON target.curation_id = source.curation_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
WHEN NOT MATCHED BY SOURCE THEN DELETE

## Verify sync results

In [ ]:
%sql
SELECT
  COUNT(*)                                        AS location_curation_rows,
  COUNT_IF(caused_by IS NOT NULL)                 AS twin_rows,
  COUNT(DISTINCT location_id)                     AS locations,
  MAX(updated_datetime)                           AS last_sync
FROM openalex.works.location_curations